# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrishaSolanki-coder/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes


The action queue ranks content pages by their predicted probability of decline. Each page also receives reason codes based on observable signals such as declining performance, low CTR, low engagement, stale content, or weak search position. The reason codes are intended to make the recommendation understandable to a content team rather than presenting the model score alone.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

# Load dataset
url = "https://raw.githubusercontent.com/PrishaSolanki-coder/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# Final model features
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

categorical_features = [
    "content_type",
    "main_intent",
    "provider_used",
    "model_used"
]

feature_columns = numeric_features + categorical_features

# Client-grouped split
clients = df["client_id"].dropna().unique()

train_clients, val_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_df = df[df["client_id"].isin(train_clients)].copy()
val_df = df[df["client_id"].isin(val_clients)].copy()

X_train = train_df[feature_columns]
y_train = train_df["is_declining_label"]

X_val = val_df[feature_columns]
y_val = val_df["is_declining_label"]

# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Logistic Regression
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

# Prediction score
val_df["decline_probability"] = model.predict_proba(
    X_val
)[:, 1]

# Rank highest-risk pages first
val_df["priority_rank"] = (
    val_df["decline_probability"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# -----------------------------
# Reason codes
# -----------------------------

def get_reason_codes(row):
    reasons = []

    if row["days_since_last_update"] >= 180:
        reasons.append("stale_content")

    if row["impressions_90d"] >= 500 and row["ctr"] < 1:
        reasons.append("low_ctr_visible_page")

    if row["impressions_90d"] >= 500 and row["sessions_90d"] < 100:
        reasons.append("low_engagement_visible_page")

    if row["avg_position"] > 10:
        reasons.append("weak_search_position")

    if row["word_count"] < 500:
        reasons.append("thin_content")

    if row["is_declining_label"] == 1:
        reasons.append("observed_decline")

    if len(reasons) == 0:
        reasons.append("model_priority")

    return ", ".join(reasons)

val_df["reason_codes"] = val_df.apply(
    get_reason_codes,
    axis=1
)

# Action recommendation
val_df["recommended_action"] = np.where(
    val_df["decline_probability"] >= 0.50,
    "refresh_review",
    "monitor"
)

# Final queue
action_queue = val_df[
    [
        "priority_rank",
        "content_id",
        "client_id",
        "decline_probability",
        "recommended_action",
        "reason_codes"
    ]
].sort_values("priority_rank")

print("Top 20 recommended actions:")
print(action_queue.head(20).to_string(index=False))

print("\nAction counts:")
print(action_queue["recommended_action"].value_counts())


Top 20 recommended actions:
 priority_rank           content_id         client_id  decline_probability recommended_action                                                                       reason_codes
             1 content_5c7a9bd6cbfe client_8527a891e2             0.770406     refresh_review                                                                     model_priority
             2 content_84d12054c0c0 client_9400f1b21c             0.769322     refresh_review                                                    stale_content, observed_decline
             3 content_38bab00f71e2 client_8527a891e2             0.767657     refresh_review                                                                     model_priority
             4 content_f488400fca67 client_9400f1b21c             0.764163     refresh_review                                                    stale_content, observed_decline
             5 content_df1fa766cac2 client_9400f1b21c             0.763731     refresh_

## 2. Intended use and limits



The queue is intended for content teams to prioritize pages for human review. A high model score means that a page is more strongly associated with the declining-content pattern in this dataset; it does not prove that the page needs a refresh.

The model is limited by the available 90-day aggregated data, the historical label definition, and the validation population. It should not be used as an automatic publishing, deletion, or content-change system. New clients, major search changes, missing data, or changes in traffic patterns may reduce its reliability.

In [2]:
print("Intended-use checks")

print("\nValidation clients:", val_df["client_id"].nunique())
print("Validation pages:", len(val_df))

print(
    "Pages recommended for refresh review:",
    (val_df["recommended_action"] == "refresh_review").sum()
)

print(
    "Pages recommended for monitoring:",
    (val_df["recommended_action"] == "monitor").sum()
)

print(
    "\nHighest model score:",
    round(val_df["decline_probability"].max(), 3)
)

print(
    "Lowest model score:",
    round(val_df["decline_probability"].min(), 3)
)


Intended-use checks

Validation clients: 7
Validation pages: 3419
Pages recommended for refresh review: 2746
Pages recommended for monitoring: 673

Highest model score: 0.77
Lowest model score: 0.081


## 3. Human review + the no-go list


Before acting on a recommendation, a content specialist should check the page itself, the search intent, current search results, recent traffic changes, content quality, business importance, and whether the page has already been updated.

The model should never automatically publish, delete, redirect, or substantially rewrite content. It should also not override a subject-matter expert when the model recommendation conflicts with important business or editorial context. The queue is a prioritization tool, not an autonomous content decision-maker.

In [3]:
# Human-review checklist for the highest-priority pages

top_review_queue = action_queue[
    action_queue["recommended_action"] == "refresh_review"
].head(20).copy()

top_review_queue["human_checks_required"] = (
    "Check search intent; inspect current content; "
    "check recent traffic; verify business importance; "
    "confirm no recent manual update"
)

print(
    top_review_queue[
        [
            "priority_rank",
            "content_id",
            "decline_probability",
            "reason_codes",
            "human_checks_required"
        ]
    ].to_string(index=False)
)

print("\nNO-GO LIST:")
print("1. Do not automatically publish changes.")
print("2. Do not automatically delete or redirect pages.")
print("3. Do not automatically rewrite content.")
print("4. Do not treat model score as proof of content failure.")
print("5. Do not skip human review.")


 priority_rank           content_id  decline_probability                                                                       reason_codes                                                                                                           human_checks_required
             1 content_5c7a9bd6cbfe             0.770406                                                                     model_priority Check search intent; inspect current content; check recent traffic; verify business importance; confirm no recent manual update
             2 content_84d12054c0c0             0.769322                                                    stale_content, observed_decline Check search intent; inspect current content; check recent traffic; verify business importance; confirm no recent manual update
             3 content_38bab00f71e2             0.767657                                                                     model_priority Check search intent; inspect current content; check rece

## 4. Monitoring / retrain triggers


The model should be reviewed if its ranking quality falls, if the distribution of important input features changes substantially, or if the relationship between the features and decline label changes. A retraining review should also be triggered when new client or time-period data becomes available and the existing model no longer represents current content behavior.

The main operational trigger is a sustained drop in Precision@50 on a recent labeled validation sample. Data-quality problems and major changes in search or content behavior should also trigger an earlier review.

In [4]:
# Current validation monitoring baseline

def precision_at_k(y_true, scores, k=50):
    temp = pd.DataFrame({
        "actual": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_k = temp.sort_values(
        "score",
        ascending=False
    ).head(k)

    return top_k["actual"].mean()

current_precision_50 = precision_at_k(
    y_val,
    val_df["decline_probability"],
    50
)

print(
    "Current Precision@50:",
    round(current_precision_50, 3)
)

print("\nMonitoring triggers:")

print(
    "1. Review if Precision@50 falls below:",
    round(max(0, current_precision_50 - 0.10), 3)
)

print("2. Review if important feature distributions change substantially.")
print("3. Review if missing-data rates increase substantially.")
print("4. Review when new labeled client/time-period data becomes available.")
print("5. Retrain after sustained performance or data-distribution changes.")


Current Precision@50: 0.54

Monitoring triggers:
1. Review if Precision@50 falls below: 0.44
2. Review if important feature distributions change substantially.
3. Review if missing-data rates increase substantially.
4. Review when new labeled client/time-period data becomes available.
5. Retrain after sustained performance or data-distribution changes.


## 5. Exports for the paper


The final ranked queue and monitoring summary will be saved under `work/outputs/`. These files provide reproducible artifacts that can be referenced in the capstone paper instead of manually copying results from the notebook.

In [5]:
import os

# Create output directory if it does not exist
os.makedirs("work/outputs", exist_ok=True)

# Save ranked action queue
queue_path = "work/outputs/content_action_queue.csv"

action_queue.to_csv(
    queue_path,
    index=False
)

# Save monitoring summary
monitoring_summary = pd.DataFrame({
    "metric": [
        "validation_clients",
        "validation_pages",
        "precision_at_50",
        "refresh_review_count",
        "monitor_count"
    ],
    "value": [
        val_df["client_id"].nunique(),
        len(val_df),
        current_precision_50,
        (val_df["recommended_action"] == "refresh_review").sum(),
        (val_df["recommended_action"] == "monitor").sum()
    ]
})

monitoring_path = "work/outputs/model_monitoring_summary.csv"

monitoring_summary.to_csv(
    monitoring_path,
    index=False
)

print("Files saved:")
print(queue_path)
print(monitoring_path)

print("\nQueue rows exported:", len(action_queue))


Files saved:
work/outputs/content_action_queue.csv
work/outputs/model_monitoring_summary.csv

Queue rows exported: 3419


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.